In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ==========================
# FIND FILES
# ==========================

files=sorted(
    Path(".")
    .glob("cleaned_*.xlsx")
)

print("Files Found:",len(files))

# ==========================
# COLUMN STANDARDIZATION
# ==========================

column_map={

"vessel_name":"name_of_ship",
"name_of_ship":"name_of_ship",

"vessel_type":"type_of_ship",
"type_of_ship":"type_of_ship",

"date_of_the_near_miss":"date_of_the_near_miss",
"date_of_occurrence":"date_of_the_near_miss",
"date_and_time_of_occurrence":"date_of_the_near_miss",

"description":"description",
"description_of_near_miss":"description",
"description_of_event_leading_to_the_incident":"description",
"incidentdetails":"description",
"incidenteventdetails":"description",
"details_of_near_miss":"description",

"incident_category":"incident_category",
"incident_category_(potential)":"incident_category",

"possible_consequence":"possible_consequence",

"corrective_action":"corrective_action",
"corrective_action(s)":"corrective_action",

"area_of_concern":"area_of_concern",

"report_date":"report_date",
"date_of_report":"report_date"
}

# ==========================
# DATE FORMATTER
# ==========================

def clean_date(v):

    if pd.isna(v):
        return np.nan

    v=str(v).strip()

    if v=="":
        return np.nan

    if v.lower() in [
        "na",
        "n/a",
        "not mentioned"
    ]:
        return np.nan

    if re.match(
        r"^[A-Za-z]{3}$",
        v
    ):
        return v

    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        v
    ):
        return v

    try:

        d=pd.to_datetime(
            v,
            dayfirst=True,
            errors="coerce"
        )

        if pd.notna(d):

            return d.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return v

# ==========================
# STANDARDIZE FILE
# ==========================

all_df=[]

for file in files:

    print("\nReading:",file.name)

    df=pd.read_excel(
        file,
        dtype=str
    )

    df.columns=(
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ","_",regex=False)
    )

    # rename similar columns

    rename={}

    for col in df.columns:

        if col in column_map:

            rename[col]=column_map[col]

    df=df.rename(
        columns=rename
    )

    # merge duplicate columns

    cols=df.columns.tolist()

    duplicates=[]

    for col in set(cols):

        same=[
            c
            for c in cols
            if c==col
        ]

        if len(same)>1:

            duplicates.append(col)

    for col in duplicates:

        temp=df.loc[:,df.columns==col]

        df[col]=(
            temp
            .fillna("")
            .astype(str)
            .apply(
                lambda x:
                " | ".join(
                    [
                        i
                        for i in x
                        if i.strip()
                    ]
                ),
                axis=1
            )
        )

        df=df.loc[
            :,
            ~df.columns.duplicated()
        ]

    # format dates

    for col in df.columns:

        if "date" in col:

            df[col]=(
                df[col]
                .apply(clean_date)
            )

    df["source_file"]=file.name

    all_df.append(df)

# ==========================
# UNION OF ALL COLUMNS
# ==========================

master_cols=[]

for df in all_df:

    for c in df.columns:

        if c not in master_cols:

            master_cols.append(c)

# ==========================
# ALIGN
# ==========================

aligned=[]

for df in all_df:

    for c in master_cols:

        if c not in df.columns:

            df[c]=np.nan

    aligned.append(
        df[
            master_cols
        ]
    )

# ==========================
# MERGE
# ==========================

final=pd.concat(
    aligned,
    ignore_index=True
)

# ==========================
# REMOVE DUPLICATES
# ==========================

dup=final.duplicated().sum()

final=final.drop_duplicates()

print("\nDuplicates Removed:",dup)

# ==========================
# RESET SI NO
# ==========================

drop=[]

for c in final.columns:

    clean=c.lower()

    if (
        "si" in clean
        or
        "sl" in clean
        or
        "serial" in clean
    ):

        drop.append(c)

final=final.drop(
    columns=drop,
    errors="ignore"
)

final.insert(
    0,
    "SI_No",
    range(
        1,
        len(final)+1
    )
)

# ==========================
# REPORT
# ==========================

missing=(
    final
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Summary:")
print(missing)

print("\nFinal Shape:")
print(final.shape)

print("\nFinal Columns:")
print(list(final.columns))

# ==========================
# EXPORT
# ==========================

output="merged_near_miss.xlsx"

final.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

Files Found: 12

Reading: cleaned_10_Near_Miss.xlsx

Reading: cleaned_11_Near_Miss.xlsx

Reading: cleaned_12_Near_Miss.xlsx


C:\Users\vinyt\AppData\Local\Temp\ipykernel_31752\2149501092.py:89: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d=pd.to_datetime(
C:\Users\vinyt\AppData\Local\Temp\ipykernel_31752\2149501092.py:89: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d=pd.to_datetime(



Reading: cleaned_1_Near_Miss.xlsx

Reading: cleaned_2_Near_Miss.xlsx

Reading: cleaned_3_Near_Miss.xlsx


C:\Users\vinyt\AppData\Local\Temp\ipykernel_31752\2149501092.py:89: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d=pd.to_datetime(



Reading: cleaned_4_Near_Miss.xlsx

Reading: cleaned_5_Near_Miss.xlsx

Reading: cleaned_6_Near_Miss.xlsx

Reading: cleaned_7_Near_Miss.xlsx

Reading: cleaned_8_Near_Miss.xlsx

Reading: cleaned_9_Near_Miss.xlsx

Duplicates Removed: 0

Missing Summary:
ref_num/report_date       12985
name_of_ship               9614
type_of_ship               9015
date_of_the_near_miss      9639
period                    11375
                          ...  
reference_no.             15023
office/vessel             15023
report_type               15023
potential_of_near_miss    15026
extended_to               15627
Length: 83, dtype: int64

Final Shape:
(15631, 85)

Final Columns:
['SI_No', 'ref_num/report_date', 'name_of_ship', 'type_of_ship', 'date_of_the_near_miss', 'period', 'status', 'location', 'port_name', 'related_department', 'description', 'immediate_action_taken', 'potential_extent_of_damage/injury', 'near_miss_potential', 'potential_damage_category', 'potential_damage_subcategory', 'proposed_c

In [2]:
import pandas as pd

# Load merged file
df=pd.read_excel(
    "merged_near_miss.xlsx"
)

# Move source_file to last column
if "source_file" in df.columns:

    df=df[
        [
            c
            for c in df.columns
            if c!="source_file"
        ]
        +
        [
            "source_file"
        ]
    ]

# Save final file
output="merged_near_miss_final.xlsx"

df.to_excel(
    output,
    index=False
)

print("Saved:",output)

Saved: merged_near_miss_final.xlsx
